# Student Dropout Risk Prediction

This notebook contains our final competition pipeline for predicting student dropout risk.

we used three types of information:
- tabular academic/demographic data
- attendance time-series features
- counsellor note text features

The final model is an ensemble of CatBoost, TF-IDF Logistic Regression, and XGBoost with class-1 threshold tuning for Macro F1.

In [2]:
# Basic imports
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42

## 1. Load data

The main training and test files are joined with attendance and counsellor note data using `student_id`.

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
attendance = pd.read_csv("Attendance_series.csv")
notes = pd.read_csv("Counsellor_notes.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Attendance:", attendance.shape)
print("Notes:", notes.shape)

print("Target distribution:")
print(train["dropout_risk"].value_counts().sort_index())

Train: (12000, 18)
Test: (3000, 17)
Attendance: (1048575, 5)
Notes: (15000, 2)
Target distribution:
dropout_risk
0    7200
1    3000
2    1800
Name: count, dtype: int64


## 2. Counsellor note aggregation

Some students may have more than one note, so we combine notes at student level.

In [4]:
student_notes = (
    notes.groupby("student_id")["counsellor_note"]
    .apply(" ".join)
    .reset_index()
)

student_notes.head()

,student_id,counsellor_note
0,STU00001,Student is performing well. Follow-up required.
1,STU00002,Needs to improve focus in class. Action plan d...
2,STU00003,"Struggling slightly with core subjects, advise..."
3,STU00004,Student expressed some stress regarding course...
4,STU00005,Multiple backlogs. Demotivated. Action plan di...


## 3. Attendance feature engineering

we create overall attendance statistics, semester-wise attendance means, semester drops, and a linear attendance slope.

The slope feature captures whether the student's attendance is improving or declining over time.

In [5]:
# Overall attendance statistics

attendance_features = attendance.groupby("student_id").agg(
    attendance_mean=("attendance_pct", "mean"),
    attendance_std=("attendance_pct", "std"),
    attendance_min=("attendance_pct", "min"),
    attendance_max=("attendance_pct", "max"),
    attendance_median=("attendance_pct", "median")
).reset_index()

# Semester-wise attendance means

sem = (
    attendance.groupby(["student_id", "semester"])["attendance_pct"]
    .mean()
    .reset_index()
    .pivot(index="student_id", columns="semester", values="attendance_pct")
    .reset_index()
)

sem.columns = [
    "student_id" if c == "student_id" else f"sem{c}_att"
    for c in sem.columns
]

attendance_features = attendance_features.merge(
    sem,
    on="student_id",
    how="left"
)

# Change in attendance from one semester to next
for a, b in [(1, 2), (2, 3), (3, 4)]:
    ca = f"sem{a}_att"
    cb = f"sem{b}_att"

    if ca in attendance_features.columns and cb in attendance_features.columns:
        attendance_features[f"att_drop_{a}_{b}"] = (
            attendance_features[cb] - attendance_features[ca]
        )

# Attendance slope over semester-week timeline
tmp = (
    attendance.groupby(["student_id", "semester", "week"])["attendance_pct"]
    .mean()
    .reset_index()
)

tmp["time"] = tmp["semester"] * 10 + tmp["week"]

def get_slope(g):
    if len(g) < 2:
        return 0
    return np.polyfit(g["time"].values, g["attendance_pct"].values, 1)[0]

slopes = (
    tmp.groupby("student_id")
    .apply(get_slope)
    .reset_index(name="attendance_slope")
)

attendance_features = attendance_features.merge(
    slopes,
    on="student_id",
    how="left"
)

print("Built attendance features")
attendance_features.head()

Built attendance features


,student_id,attendance_mean,attendance_std,attendance_min,attendance_max,attendance_median,sem1_att,sem2_att,sem3_att,att_drop_1_2,att_drop_2_3,attendance_slope
0,STU00001,0.806669,0.214452,0.1276,1.0,0.85380,0.821704,0.791467,0.806850,-0.030238,0.015383,0.000755
1,STU00002,0.913447,0.175171,0.1816,1.0,1.00000,0.959000,0.871404,0.909618,-0.087596,0.038214,-0.002753
2,STU00003,0.743041,0.209314,0.1237,1.0,0.78905,0.742754,0.751625,0.733991,0.008871,-0.017634,0.000199
3,STU00004,0.781174,0.191048,0.1394,1.0,0.83035,0.800104,0.797804,0.742382,-0.002300,-0.055422,-0.001741
4,STU00005,0.782371,0.183122,0.1460,1.0,0.81750,0.765017,0.785500,0.797891,0.020483,0.012391,0.002656


## 4. Merge all data

Now we join notes and attendance features with train/test.

In [6]:
train_f = train.merge(student_notes, on="student_id", how="left")
test_f = test.merge(student_notes, on="student_id", how="left")

train_f = train_f.merge(attendance_features, on="student_id", how="left")
test_f = test_f.merge(attendance_features, on="student_id", how="left")

# Binning attendance slope helped because the relationship was not perfectly linear.

train_f["attendance_slope_bin"] = pd.qcut(
    train_f["attendance_slope"],
    10,
    labels=False,
    duplicates="drop"
)

bins = np.quantile(
    train_f["attendance_slope"].fillna(0),
    np.linspace(0, 1, 11)
)
bins = np.unique(bins)

test_f["attendance_slope_bin"] = pd.cut(
    test_f["attendance_slope"].fillna(0),
    bins=bins,
    labels=False,
    include_lowest=True
)

test_f["attendance_slope_bin"] = test_f["attendance_slope_bin"].fillna(0)

print("Merged data")
print(train_f.shape, test_f.shape)

Merged data
(12000, 31) (3000, 30)


## 5. Tabular and text-based feature engineering

Important engineered features:
- CGPA mean/trend/drop
- backlog total/trend
- income and scholarship interaction
- attendance decline flags
- counsellor note keyword indicators

In [7]:
cgpa_cols = ["cgpa_sem1", "cgpa_sem2", "cgpa_sem3", "cgpa_sem4"]
backlog_cols = ["backlogs_sem1", "backlogs_sem2", "backlogs_sem3"]

for df in [train_f, test_f]:

    df["counsellor_note"] = df["counsellor_note"].fillna("Missing").astype(str)

    # Academic performance features
    df["cgpa_mean"] = df[cgpa_cols].mean(axis=1)
    df["cgpa_std"] = df[cgpa_cols].std(axis=1)
    df["cgpa_min"] = df[cgpa_cols].min(axis=1)
    df["cgpa_max"] = df[cgpa_cols].max(axis=1)
    df["cgpa_range"] = df["cgpa_max"] - df["cgpa_min"]

    df["cgpa_s12"] = df["cgpa_sem2"] - df["cgpa_sem1"]
    df["cgpa_s23"] = df["cgpa_sem3"] - df["cgpa_sem2"]
    df["cgpa_s34"] = df["cgpa_sem4"] - df["cgpa_sem3"]
    df["cgpa_trend"] = df["cgpa_sem4"] - df["cgpa_sem1"]
    df["worst_cgpa_drop"] = df[["cgpa_s12", "cgpa_s23", "cgpa_s34"]].min(axis=1)

    # Backlog features
    df["backlog_total"] = df[backlog_cols].sum(axis=1)
    df["backlog_max"] = df[backlog_cols].max(axis=1)
    df["backlog_trend"] = df["backlogs_sem3"] - df["backlogs_sem1"]

    # Income and scholarship interaction
    df["low_income"] = (df["family_income"] == "Low").astype(int)
    df["medium_income"] = (df["family_income"] == "Medium").astype(int)
    df["high_income"] = (df["family_income"] == "High").astype(int)

    df["income_scholarship"] = (
        df["family_income"].astype(str) + "_" + df["scholarship"].astype(str)
    )

    df["low_no_scholarship"] = (
        (df["family_income"] == "Low") & (df["scholarship"] == 0)
    ).astype(int)

    df["high_with_scholarship"] = (
        (df["family_income"] == "High") & (df["scholarship"] == 1)
    ).astype(int)

    df["scholarship_x_cgpa"] = df["scholarship"] * df["cgpa_mean"]
    df["scholarship_x_backlog"] = df["scholarship"] * df["backlog_total"]

    # Attendance decline features
    df["attendance_declining"] = (df["attendance_slope"] < 0).astype(int)
    df["severe_attendance_decline"] = (df["attendance_slope"] < -0.003).astype(int)

    df["parttime_x_attendance"] = df["part_time_job"] * df["attendance_mean"]
    df["parttime_x_slope"] = df["part_time_job"] * df["attendance_slope"]
    df["parttime_x_backlog"] = df["part_time_job"] * df["backlog_total"]

    df["cgpa_x_attendance"] = df["cgpa_mean"] * df["attendance_mean"]
    df["cgpa_x_slope"] = df["cgpa_mean"] * df["attendance_slope"]

    # Counsellor note keyword flags
    df["note_action_plan"] = df["counsellor_note"].str.contains(
        "Action plan discussed", case=False, na=False
    ).astype(int)

    df["note_followup"] = df["counsellor_note"].str.contains(
        "Follow-up required", case=False, na=False
    ).astype(int)

    df["note_monitored"] = df["counsellor_note"].str.contains(
        "Situation monitored", case=False, na=False
    ).astype(int)

    df["note_high"] = (
        df["counsellor_note"].str.contains("dropping out", case=False, na=False).astype(int)
        + df["counsellor_note"].str.contains("financial stress", case=False, na=False).astype(int)
        + df["counsellor_note"].str.contains("mental health", case=False, na=False).astype(int)
        + df["counsellor_note"].str.contains("demotivated", case=False, na=False).astype(int)
        + df["counsellor_note"].str.contains("severe lack", case=False, na=False).astype(int)
        + df["counsellor_note"].str.contains("emergency contact", case=False, na=False).astype(int)
    )

    df["note_low"] = (
        df["counsellor_note"].str.contains("performing well", case=False, na=False).astype(int)
        + df["counsellor_note"].str.contains("good academic-social", case=False, na=False).astype(int)
        + df["counsellor_note"].str.contains("participates actively", case=False, na=False).astype(int)
        + df["counsellor_note"].str.contains("no major issues", case=False, na=False).astype(int)
        + df["counsellor_note"].str.contains("no further action", case=False, na=False).astype(int)
    )

print("Feature engineering done")

Feature engineering done


## 6. Clean missing values

Categorical values are converted to strings for CatBoost. Numeric missing values are filled with median.

In [8]:
cat_cols = [
    "branch",
    "gender",
    "hostel_status",
    "family_income",
    "parent_education",
    "counsellor_note",
    "income_scholarship",
    "attendance_slope_bin"
]

for df in [train_f, test_f]:

    for col in cat_cols:
        df[col] = df[col].fillna("Missing").astype(str)

    for col in df.columns:
        if col in cat_cols:
            continue

        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna("Missing")

print("Train missing:", train_f.isnull().sum().sum())
print("Test missing:", test_f.isnull().sum().sum())

Train missing: 0
Test missing: 0


## 7. Prepare model matrices

we keep categorical features as strings for CatBoost, and separately label-encode them for XGBoost.

In [9]:
X = train_f.drop(["student_id", "dropout_risk"], axis=1)
y = train_f["dropout_risk"]

X_test = test_f.drop(["student_id"], axis=1)

text_train = train_f["counsellor_note"].fillna("")
text_test = test_f["counsellor_note"].fillna("")

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

cat_oof = np.zeros((len(X), 3))
text_oof = np.zeros((len(X), 3))
xgb_oof = np.zeros((len(X), 3))

cat_test = np.zeros((len(X_test), 3))
text_test_prob = np.zeros((len(X_test), 3))
xgb_test = np.zeros((len(X_test), 3))

# XGBoost needs numeric categorical encoding.

X_xgb = X.copy()
X_test_xgb = X_test.copy()

for col in cat_cols:
    le = LabelEncoder()

    all_values = pd.concat(
        [X_xgb[col].astype(str), X_test_xgb[col].astype(str)],
        axis=0
    )

    le.fit(all_values)

    X_xgb[col] = le.transform(X_xgb[col].astype(str))
    X_test_xgb[col] = le.transform(X_test_xgb[col].astype(str))

print("Model matrices ready")

Model matrices ready


## 8. Train CatBoost, TF-IDF Logistic Regression, and XGBoost

we use 5-fold stratified validation and store out-of-fold probabilities for each model.

These OOF probabilities are later used for ensemble optimization.

In [10]:
for fold, (tr,val) in enumerate(cv.split(X, y)):

    print("\nFold", fold + 1)

   
    # CatBoost on tabular features
    
    cat = CatBoostClassifier(
        iterations=2500,
        depth=8,
        learning_rate=0.02,
        l2_leaf_reg=8,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        class_weights=[1.0, 1.3, 1.9],
        random_seed=RANDOM_STATE + fold,
        verbose=False
    )

    cat.fit(
        X.iloc[tr],
        y.iloc[tr],
        cat_features=cat_cols
    )

    cat_oof[val] = cat.predict_proba(X.iloc[val])
    cat_test += cat.predict_proba(X_test) / 5

    cat_pred = np.argmax(cat_oof[val], axis=1)
    print("CatBoost F1:", f1_score(y.iloc[val], cat_pred, average="macro"))

    
    # TF-IDF + Logistic Regression on counsellor notes
   
    tfidf = TfidfVectorizer(
        max_features=100000,
        ngram_range=(1, 5),
        min_df=1,
        sublinear_tf=True
    )

    Xtr_text = tfidf.fit_transform(text_train.iloc[tr])
    Xval_text = tfidf.transform(text_train.iloc[val])
    Xte_text = tfidf.transform(text_test)

    logreg = LogisticRegression(
        max_iter=10000,
        C=10,
        class_weight="balanced"
    )

    logreg.fit(Xtr_text, y.iloc[tr])

    text_oof[val] = logreg.predict_proba(Xval_text)
    text_test_prob += logreg.predict_proba(Xte_text) / 5

    text_pred = np.argmax(text_oof[val], axis=1)
    print("Text F1:", f1_score(y.iloc[val], text_pred, average="macro"))

   
    # XGBoost on encoded tabular features
    
    xgb = XGBClassifier(
        n_estimators=900,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.85,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE + fold,
        n_jobs=-1
    )

    xgb.fit(X_xgb.iloc[tr], y.iloc[tr])

    xgb_oof[val] = xgb.predict_proba(X_xgb.iloc[val])
    xgb_test += xgb.predict_proba(X_test_xgb) / 5

    xgb_pred = np.argmax(xgb_oof[val], axis=1)
    print("XGB F1:", f1_score(y.iloc[val], xgb_pred, average="macro"))

print("\nBase model OOF scores:")
print("CatBoost OOF:", f1_score(y, np.argmax(cat_oof, axis=1), average="macro"))
print("Text OOF:", f1_score(y, np.argmax(text_oof, axis=1), average="macro"))
print("XGB OOF:", f1_score(y, np.argmax(xgb_oof, axis=1), average="macro"))


Fold 1
CatBoost F1: 0.7092405571066397
Text F1: 0.6930025406478263
XGB F1: 0.7051525330163101

Fold 2
CatBoost F1: 0.7075770206099005
Text F1: 0.6849568512406795
XGB F1: 0.6905683422713582

Fold 3
CatBoost F1: 0.7100546468830359
Text F1: 0.6958468296217433
XGB F1: 0.6974936845146688

Fold 4
CatBoost F1: 0.7098522604548139
Text F1: 0.7019622605655415
XGB F1: 0.6975195337831749

Fold 5
CatBoost F1: 0.7150445584589481
Text F1: 0.6858523147275268
XGB F1: 0.7107390205291174

Base model OOF scores:
CatBoost OOF: 0.7104002077474337
Text OOF: 0.6923332339395493
XGB OOF: 0.7002963708578895


## 9. Ensemble weight and class-1 threshold search

The main issue was class 1 being the middle/gray zone.
So after combining model probabilities, We tune a threshold for class 1 to improve Macro F1.

In [11]:
best_score = 0
best_weights = None
best_t = None

# Grid search for CatBoost, Text, and XGBoost weights

for wc in np.arange(0.45, 0.76, 0.05):
    for wt in np.arange(0.10, 0.41, 0.05):

        wx = 1 - wc - wt

        if wx < 0:
            continue

        prob = (
            wc * cat_oof
            + wt * text_oof
            + wx * xgb_oof
        )

        # Tune class 1 threshold because class 1 is the hardest class.
        
        for t in np.arange(0.42, 0.47, 0.001):

            pred = []

            for p in prob:
                if p[1] > t:
                    pred.append(1)
                else:
                    pred.append(np.argmax(p))

            score = f1_score(y, pred, average="macro")

            if score > best_score:
                best_score = score
                best_weights = (wc, wt, wx)
                best_t = t

print("BEST WEIGHTS:", best_weights)
print("BEST THRESHOLD:", best_t)
print("BEST OOF:", best_score)

BEST WEIGHTS: (np.float64(0.55), np.float64(0.3500000000000001), np.float64(0.09999999999999987))
BEST THRESHOLD: 0.447
BEST OOF: 0.7213610148144184


## 10. Validation report

This gives the final confusion matrix and class-wise report using the best ensemble found above.

In [12]:
oof_prob = (
    best_weights[0] * cat_oof
    + best_weights[1] * text_oof
    + best_weights[2] * xgb_oof
)

oof_pred = []

for p in oof_prob:
    if p[1] > best_t:
        oof_pred.append(1)
    else:
        oof_pred.append(np.argmax(p))

oof_pred = np.array(oof_pred)

print("Final OOF Macro F1:", f1_score(y, oof_pred, average="macro"))
print("\nConfusion Matrix:")
print(confusion_matrix(y, oof_pred))
print("\nClassification Report:")
print(classification_report(y, oof_pred))

Final OOF Macro F1: 0.7213610148144184

Confusion Matrix:
[[6155 1037    8]
 [ 734 1691  575]
 [  15  390 1395]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.85      0.87      7200
           1       0.54      0.56      0.55      3000
           2       0.71      0.78      0.74      1800

    accuracy                           0.77     12000
   macro avg       0.71      0.73      0.72     12000
weighted avg       0.78      0.77      0.77     12000



## 11. Create final submission

we use the same ensemble weights and threshold found from OOF validation.

In [14]:
test_prob = (
    best_weights[0] * cat_test
    + best_weights[1] * text_test_prob
    + best_weights[2] * xgb_test
)

final_pred = []

for p in test_prob:
    if p[1] > best_t:
        final_pred.append(1)
    else:
        final_pred.append(np.argmax(p))

final_pred = np.array(final_pred)

submission = pd.DataFrame({
    "student_id": test["student_id"],
    "dropout_risk": final_pred
})

submission.to_csv("Final_pipeline.csv", index=False)

print("Saved submission_final.csv")
print(submission.head())
print("\nSubmission distribution:")
print(submission["dropout_risk"].value_counts().sort_index())

Saved submission_final.csv
  student_id  dropout_risk
0   STU03679             0
1   STU11070             0
2   STU13561             1
3   STU00061             1
4   STU02416             2

Submission distribution:
dropout_risk
0    1718
1     776
2     506
Name: count, dtype: int64


## Notes

The most useful signals found during experimentation were:

- counsellor notes through TF-IDF
- attendance deterioration through attendance slope
- income-scholarship interaction
- class-1 threshold tuning to handle the middle-risk group

We did not use `solution (1).csv` for training or tuning because it appears to contain target-like information for test rows.